# Eval checkpoint — metric CHÍNH THỨC (đúng protocol paper)

Load weight đã train (trên Drive) → chạy `test()` của code → ra **Recall/HR/NDCG** đúng cách đo của paper.
**KHÔNG train lại, chạy CPU được** (eval nhẹ). Dùng số này để đối chiếu paper (KHÔNG dùng số bảng demo).

## 1. Setup

In [ ]:
!pip install -q absl-py setproctitle deprecated faiss-cpu
import os, glob, re, torch
REPO='/content/DIVRS'
if os.path.exists(REPO):
    !cd {REPO} && git fetch -q origin && git reset --hard -q origin/main
else:
    !git clone -q https://github.com/HatDuaa/DIVRS-reproduce.git {REPO}
%cd {REPO}
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/ml10m/output/train_coo_record.npz'):
    !cd data && wget -q https://raw.githubusercontent.com/tsinghua-fib-lab/DICE/main/data/ml10m.zip && unzip -oq ml10m.zip
from google.colab import drive; drive.mount('/content/drive')
OUT_DIR='/content/drive/MyDrive/Colab Notebooks/DIVRS_output'
!git log --oneline -1

## 2. Tìm checkpoint DIVRS (epoch cao nhất trên Drive)

In [ ]:
runs=[r for r in glob.glob(OUT_DIR+'/*/') if 'divrs' in r.lower() and glob.glob(r+'ckpt/epoch_*.pth')]
run=max(runs,key=lambda r:len(glob.glob(r+'ckpt/epoch_*.pth')))
cks=glob.glob(run+'ckpt/epoch_*.pth')
CKPT=max(cks,key=lambda x:int(re.findall(r'epoch_(\d+)',x)[0]))
print('Run:', os.path.basename(run.rstrip('/')))
print('So checkpoint:', len(cks), '| dung epoch cao nhat:', os.path.basename(CKPT))

## 3. Chạy TEST chính thức trên checkpoint đó
Tạo output mới `eval_out/`. Eval nhẹ (chỉ rank embedding), CPU vài phút.

In [ ]:
USE_GPU=torch.cuda.is_available(); print('use_gpu =', USE_GPU)
!python app.py \
  --flagfile config/ml10m_DIVRS.cfg \
  --test=True --test_ckpt="{CKPT}" \
  --output ./eval_out/ \
  --use_gpu={USE_GPU} --gpu_id=0 \
  --cg_use_gpu=False --num_workers=2

## 4. In metric chính thức (Recall / HR / NDCG mọi Top-K)

In [ ]:
ev=max(glob.glob('eval_out/*/'), key=os.path.getmtime)
print('Eval run:', ev)
!find "{ev}test_log" -type f -exec grep -hE "TEST results|recall|hit_ratio|ndcg" {{}} + 2>/dev/null

## Ghi chú
- Đây là số **đúng protocol code/paper** — dùng cho báo cáo.
- Đối chiếu paper DIVRS (MF, ML-10M): **Recall@20=0.1691, HR@20=0.5302, NDCG@20=0.1128**.
- Nếu số ở đây gần các giá trị đó → tái lập thành công. Thấp hơn → cần train thêm epoch.